In [3]:
import numpy as np
import pandas as pd
import joblib

# Tree-based Machine Learning Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    classification_report, 
    confusion_matrix
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# Load preprocessed arrays from disk
X_train = np.load('data/processed/X_train_tab.npy')
X_val   = np.load('data/processed/X_val_tab.npy')
X_test  = np.load('data/processed/X_test_tab.npy')

y_train = np.load('data/processed/y_train_tab.npy')
y_val   = np.load('data/processed/y_val_tab.npy')
y_test  = np.load('data/processed/y_test_tab.npy')

feature_cols = pd.read_csv('data/processed/feature_names.csv')['0'].tolist()

print(f"Train Shape: {X_train.shape}")
print(f"Validation Shape: {X_val.shape}")
print(f"Test Shape: {X_test.shape}")

Train Shape: (25564, 21)
Validation Shape: (5478, 21)
Test Shape: (5478, 21)


In [5]:
# 1. Initialize Models
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=12, 
    random_state=42, 
    n_jobs=-1
)

gbm_model = GradientBoostingClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42
)

# 2. Fit Models on Training Data
print("Training Random Forest Classifier...")
rf_model.fit(X_train, y_train)

print("Training Gradient Boosting Machine (GBM)...")
gbm_model.fit(X_train, y_train)

print("Model training complete.")

Training Random Forest Classifier...
Training Gradient Boosting Machine (GBM)...
Model training complete.


In [6]:
def evaluate_model(model, X, y, model_name="Model"):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else y_pred
    
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    auc = roc_auc_score(y, y_prob)
    
    print(f"=== {model_name} Evaluation Metrics ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {auc:.4f}\n")
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1, "ROC-AUC": auc}

rf_results = evaluate_model(rf_model, X_test, y_test, "Random Forest")
gbm_results = evaluate_model(gbm_model, X_test, y_test, "Gradient Boosting Machine")

=== Random Forest Evaluation Metrics ===
Accuracy:  0.9995
Precision: 1.0000
Recall:    0.9982
F1-Score:  0.9991
ROC-AUC:   1.0000

=== Gradient Boosting Machine Evaluation Metrics ===
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000
ROC-AUC:   1.0000



In [7]:
import os
os.makedirs('models/saved_models', exist_ok=True)

# Save best performing model binaries
joblib.dump(rf_model, 'models/saved_models/random_forest_env_model.pkl')
joblib.dump(gbm_model, 'models/saved_models/gbm_env_model.pkl')

print("Trained models saved to 'models/saved_models/'.")

Trained models saved to 'models/saved_models/'.
